# Augmentation Visualizer

Visualises each candidate appearance augmentation on a synthetic 224x224 grayscale image.
No TensorFlow/Keras required -- pure NumPy + PIL.

Categories: **Tone | Noise | Random Lines | Edge Effects | Blur | Random Profiles**

---

## Table of Contents

| # | Category | Augmentations |
|---|---|---|
| 1 | [Tone](#cat-tone) | Gamma, Exposure simulation, Fog/haze |
| 2 | [Noise](#cat-noise) | Salt-and-pepper |
| 3 | [Random Lines](#cat-lines) | Horizontal lines, Vertical lines |
| 4 | [Edge Effects](#cat-edge) | Horizontal fading, Vertical fading, Horizontal bilateral, Vertical bilateral |
| 5 | [Blur](#cat-blur) | Gaussian blur |
| 6 | [Random Profiles](#cat-random) | Horizontal profile, Vertical profile |
| -- | [Combination](#cat-combo) | Chained augmentations |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter

from augmentations import *

## Synthetic image

Noisy mid-grey background with three solid rectangles of different tones.

In [ ]:
def make_synthetic(seed: int = 42) -> np.ndarray:
    """Return a 224x224 uint8 grayscale array."""
    rng = np.random.default_rng(seed)
    img = rng.integers(110, 145, (224, 224), dtype=np.uint8)
    rects = [
        (20,  30,  110, 120, 200),
        (130, 50,  210, 150,  50),
        (60,  140, 170, 200, 140),
    ]
    for x1, y1, x2, y2, shade in rects:
        img[y1:y2, x1:x2] = shade
    return img

ORIGINAL = make_synthetic()

plt.figure(figsize=(3, 3))
plt.imshow(ORIGINAL, cmap='gray', vmin=0, vmax=255)
plt.title('Synthetic original')
plt.axis('off')
plt.tight_layout()
plt.show()

## Helper: side-by-side comparison

In [ ]:
def show_variants(original: np.ndarray, variants: list, title: str):
    """Show original + variants in a single row. variants: list of (label, ndarray)."""
    n = 1 + len(variants)
    fig, axes = plt.subplots(1, n, figsize=(n * 3, 3))
    axes[0].imshow(original, cmap='gray', vmin=0, vmax=255)
    axes[0].set_title('original')
    axes[0].axis('off')
    for ax, (label, img) in zip(axes[1:], variants):
        ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        ax.set_title(label)
        ax.axis('off')
    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

def show_grid(fn, param_name: str, param_values: list, seeds: list, title: str):
    """Show a grid: rows=param values, cols=seeds."""
    rows, cols = len(param_values), len(seeds)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    if rows == 1:
        axes = [axes]
    for r, val in enumerate(param_values):
        for c, seed in enumerate(seeds):
            axes[r][c].imshow(fn(ORIGINAL, **{param_name: val}, seed=seed),
                              cmap='gray', vmin=0, vmax=255)
            axes[r][c].set_title(f'{param_name}={val}  seed={seed}', fontsize=8)
            axes[r][c].axis('off')
    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

<a id="cat-tone"></a>
# Category 1: Tone

Augmentations that shift overall pixel intensity - gamma curves, exposure, and haze.

---
## 1.1 Gamma correction

Non-linear tone curve: out = 255 * (in/255)^gamma.
gamma < 1 brightens shadows; gamma > 1 darkens them.

In [ ]:
show_variants(ORIGINAL, [
    ('gamma = 0.3', aug_gamma(ORIGINAL, 0.3)),
    ('gamma = 0.7', aug_gamma(ORIGINAL, 0.7)),
    ('gamma = 1.5', aug_gamma(ORIGINAL, 1.5)),
    ('gamma = 2.5', aug_gamma(ORIGINAL, 2.5)),
], 'Gamma correction')

---
## 1.2 Exposure simulation

Multiply pixel values by 2^stops - mimics opening/closing a camera aperture by N f-stops.

In [ ]:
show_variants(ORIGINAL, [
    ('stops = -2', aug_exposure(ORIGINAL, -2)),
    ('stops = -1', aug_exposure(ORIGINAL, -1)),
    ('stops = +1', aug_exposure(ORIGINAL, +1)),
    ('stops = +2', aug_exposure(ORIGINAL, +2)),
], 'Exposure simulation')

---
## 1.3 Fog / haze overlay

Blend the image with white (255) - simulates uniform haze or overexposed ambient light.

In [ ]:
show_variants(ORIGINAL, [
    ('strength = 0.1', aug_fog(ORIGINAL, 0.1)),
    ('strength = 0.3', aug_fog(ORIGINAL, 0.3)),
    ('strength = 0.5', aug_fog(ORIGINAL, 0.5)),
    ('strength = 0.7', aug_fog(ORIGINAL, 0.7)),
], 'Fog / haze overlay')

<a id="cat-noise"></a>
# Category 2: Noise

Augmentations that add pixel-level random noise to the image.

---
## 2.1 Salt-and-pepper noise

Randomly set a fraction of pixels to 0 (pepper) or 255 (salt).

In [ ]:
show_variants(ORIGINAL, [
    ('density = 0.02', aug_salt_pepper(ORIGINAL, 0.02)),
    ('density = 0.05', aug_salt_pepper(ORIGINAL, 0.05)),
    ('density = 0.10', aug_salt_pepper(ORIGINAL, 0.10)),
    ('density = 0.20', aug_salt_pepper(ORIGINAL, 0.20)),
], 'Salt-and-pepper noise')

<a id="cat-lines"></a>
# Category 3: Random Lines

Random horizontal or vertical stripes with variable width and brightness - simulates sensor artifacts or physical scratches.

---
## 3.1 Random horizontal lines

N randomly placed horizontal stripes with random width (1-3 px) and random brightness.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, ax in enumerate(axes[0]):
    ax.imshow(aug_lines_h(ORIGINAL, n_lines=3, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=3  seed={i}', fontsize=9)
    ax.axis('off')
for i, ax in enumerate(axes[1]):
    ax.imshow(aug_lines_h(ORIGINAL, n_lines=8, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=8  seed={i}', fontsize=9)
    ax.axis('off')
fig.suptitle('Random horizontal lines', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3.2 Random vertical lines

Same as above but drawn column-wise.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, ax in enumerate(axes[0]):
    ax.imshow(aug_lines_v(ORIGINAL, n_lines=3, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=3  seed={i}', fontsize=9)
    ax.axis('off')
for i, ax in enumerate(axes[1]):
    ax.imshow(aug_lines_v(ORIGINAL, n_lines=8, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=8  seed={i}', fontsize=9)
    ax.axis('off')
fig.suptitle('Random vertical lines', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id="cat-edge"></a>
# Category 4: Edge Effects

Directional fading and bilateral brightening/darkening along image edges.

---
## 4.1 Horizontal fading

Apply a linear ramp so one side of the image fades toward a target tone.

In [ ]:
show_variants(ORIGINAL, [
    ('right->black', aug_fade_horizontal(ORIGINAL, fade_to=0,   side='right')),
    ('right->white', aug_fade_horizontal(ORIGINAL, fade_to=255, side='right')),
    ('left->black',  aug_fade_horizontal(ORIGINAL, fade_to=0,   side='left')),
    ('left->white',  aug_fade_horizontal(ORIGINAL, fade_to=255, side='left')),
], 'Horizontal fading')

---
## 4.2 Vertical fading

Same as horizontal fading but applied row-wise (top / bottom).

In [ ]:
show_variants(ORIGINAL, [
    ('bottom->black', aug_fade_vertical(ORIGINAL, fade_to=0,   side='bottom')),
    ('bottom->white', aug_fade_vertical(ORIGINAL, fade_to=255, side='bottom')),
    ('top->black',    aug_fade_vertical(ORIGINAL, fade_to=0,   side='top')),
    ('top->white',    aug_fade_vertical(ORIGINAL, fade_to=255, side='top')),
], 'Vertical fading')

---
## 4.3 Horizontal bilateral brightening

Both left and right edges fade toward a brighter/darker tone with Gaussian falloff - centre stays original.

In [ ]:
show_variants(ORIGINAL, [
    ('10% bright', aug_brighten_edges_h(ORIGINAL, fade_to=255, edge_fraction=0.10)),
    ('20% bright', aug_brighten_edges_h(ORIGINAL, fade_to=255, edge_fraction=0.20)),
    ('10% dark',   aug_brighten_edges_h(ORIGINAL, fade_to=0,   edge_fraction=0.10)),
    ('20% dark',   aug_brighten_edges_h(ORIGINAL, fade_to=0,   edge_fraction=0.20)),
], 'Horizontal bilateral brightening / darkening (Gaussian)')

---
## 4.4 Vertical bilateral brightening

Both top and bottom edges fade toward a brighter/darker tone with Gaussian falloff - centre stays original.

In [ ]:
show_variants(ORIGINAL, [
    ('10% bright', aug_brighten_edges_v(ORIGINAL, fade_to=255, edge_fraction=0.10)),
    ('20% bright', aug_brighten_edges_v(ORIGINAL, fade_to=255, edge_fraction=0.20)),
    ('10% dark',   aug_brighten_edges_v(ORIGINAL, fade_to=0,   edge_fraction=0.10)),
    ('20% dark',   aug_brighten_edges_v(ORIGINAL, fade_to=0,   edge_fraction=0.20)),
], 'Vertical bilateral brightening / darkening (Gaussian)')

<a id="cat-blur"></a>
# Category 5: Blur

Spatial smoothing applied uniformly across the image.

---
## 5.1 Gaussian blur

Smooth the image using a Gaussian kernel - simulates defocus or motion at a very fine scale.

In [ ]:
show_variants(ORIGINAL, [
    ('radius = 1',  aug_blur(ORIGINAL, 1)),
    ('radius = 3',  aug_blur(ORIGINAL, 3)),
    ('radius = 6',  aug_blur(ORIGINAL, 6)),
    ('radius = 10', aug_blur(ORIGINAL, 10)),
], 'Gaussian blur')

<a id="cat-random"></a>
# Category 6: Random Profiles

Smooth random brightness waves along the horizontal or vertical axis,
built from a sum of Gaussians at random positions.

---
## 6.1 Random brightness profile - horizontal

A random smooth brightness wave applied column-wise. Built from N Gaussians at random
positions with random amplitudes - produces a complex but natural-looking variation.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, ax in enumerate(axes[0]):
    ax.imshow(aug_random_profile_h(ORIGINAL, n_changes=3, max_delta=60, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=3  seed={i}', fontsize=9)
    ax.axis('off')
for i, ax in enumerate(axes[1]):
    ax.imshow(aug_random_profile_h(ORIGINAL, n_changes=7, max_delta=60, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=7  seed={i}', fontsize=9)
    ax.axis('off')
fig.suptitle('Random brightness profile - horizontal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6.2 Random brightness profile - vertical

Same idea applied row-wise (top to bottom).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, ax in enumerate(axes[0]):
    ax.imshow(aug_random_profile_v(ORIGINAL, n_changes=3, max_delta=60, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=3  seed={i}', fontsize=9)
    ax.axis('off')
for i, ax in enumerate(axes[1]):
    ax.imshow(aug_random_profile_v(ORIGINAL, n_changes=7, max_delta=60, seed=i), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n=7  seed={i}', fontsize=9)
    ax.axis('off')
fig.suptitle('Random brightness profile - vertical', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id="cat-combo"></a>
# Combination - chained augmentations

Apply augmentations from different categories in sequence to see cumulative effect.

In [ ]:
steps = [
    ('original',          ORIGINAL),
    ('+gamma 0.7',        aug_gamma(ORIGINAL, 0.7)),
    ('+fog 0.2',          aug_fog(aug_gamma(ORIGINAL, 0.7), 0.2)),
    ('+salt&pepper 0.03', aug_salt_pepper(aug_fog(aug_gamma(ORIGINAL, 0.7), 0.2), 0.03)),
    ('+blur r=2',         aug_blur(aug_salt_pepper(aug_fog(aug_gamma(ORIGINAL, 0.7), 0.2), 0.03), 2)),
    ('+profile h',        aug_random_profile_h(aug_blur(aug_salt_pepper(aug_fog(aug_gamma(ORIGINAL, 0.7), 0.2), 0.03), 2), n_changes=4, max_delta=40)),
]

fig, axes = plt.subplots(1, len(steps), figsize=(len(steps) * 3, 3))
for ax, (label, img) in zip(axes, steps):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    ax.set_title(label, fontsize=8)
    ax.axis('off')
fig.suptitle('Chained augmentations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()